# Tutorial 1: Extracting Goal-Kick Build-Ups

This tutorial demonstrates how to extract opponent goal-kick build-up windows from SkillCorner tracking data. You'll learn:

- How to load raw match data
- How goal-kick detection works
- How build-up windows are defined
- How to inspect extracted build-ups

## Prerequisites

Ensure you have:
- SkillCorner data in `data/raw/RealMadrid/`
- Python environment with all dependencies installed

## Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import matplotlib.pyplot as plt

# Project imports
from src.extract.services.data_loader import load_match_data
from src.extract.services.goal_kick_detector import GoalKickDetector
from src.extract.services.build_up_detector import BuildUpDetector
from src.extract.services.team_utils import find_real_madrid_and_opponent

# Configuration
RAW_ROOT = Path("data/raw/RealMadrid")
MATCH_ID = "2014987"  # Example match ID

## Step 1: Load Match Data

Load the three data sources for a match:
- **Meta**: Match metadata (teams, stadium, etc.)
- **Events**: Event stream (passes, shots, etc.)
- **Tracking**: Player position tracking (25 Hz)

In [ ]:
# Load match data
meta, events, tracking = load_match_data(RAW_ROOT, MATCH_ID)

print(f"Match: {meta['home_team']['name']} vs {meta['away_team']['name']}")
print(f"Events: {len(events)} records")
print(f"Tracking: {len(tracking)} frames")
print(f"Tracking columns: {tracking.columns.tolist()[:10]}...")  # First 10 columns

## Step 2: Identify Teams

Determine which team is Real Madrid and which is the opponent.

In [ ]:
rm_team_id, opp_team_id = find_real_madrid_and_opponent(meta)

print(f"Real Madrid Team ID: {rm_team_id}")
print(f"Opponent Team ID: {opp_team_id}")

## Step 3: Detect Goal Kicks

Use `GoalKickDetector` to find all opponent goal-kick events.

In [ ]:
detector = GoalKickDetector()
goal_kicks = detector.detect(events, opp_team_id)

print(f"Found {len(goal_kicks)} opponent goal kicks")
print("\nFirst 5 goal kicks:")
print(goal_kicks.head())

## Step 4: Extract Build-Up Windows

For each goal kick, extract the build-up tracking window (setup → kick → outcome).

In [ ]:
from src.extract.services.config import ExtractionConfig

config = ExtractionConfig()
build_up_detector = BuildUpDetector(config)

# Extract first build-up as example
first_gk = goal_kicks.iloc[0]
print(f"Processing goal kick at {first_gk['time']} (period {first_gk['period']})")

# Extract build-up window
build_up_data = build_up_detector.extract(
    tracking=tracking,
    goal_kick_event=first_gk,
    rm_team_id=rm_team_id,
    opp_team_id=opp_team_id
)

if build_up_data is not None:
    window_df, metadata = build_up_data
    print(f"\nExtracted window: {len(window_df)} frames")
    print(f"Time range: {window_df['time'].min()} → {window_df['time'].max()}")
    print(f"\nMetadata:")
    for key, val in metadata.items():
        print(f"  {key}: {val}")
else:
    print("Build-up extraction failed (likely missing tracking data)")

## Step 5: Visualize Build-Up Window Phases

The build-up window consists of three phases:
1. **Setup**: From goal kick event detection to ready time (GK has ball)
2. **Kick**: From ready time to kick time (GK kicks ball)
3. **Post-kick**: From kick time to window end (5 seconds after kick)

In [ ]:
if build_up_data is not None:
    from src.features.services.utils import time_to_seconds
    
    ready_t = time_to_seconds(str(metadata['ready_time']))
    kick_t = time_to_seconds(str(metadata['kick_time']))
    
    # Add time_seconds column for plotting
    window_df['time_seconds'] = window_df['time'].apply(time_to_seconds)
    
    # Count players per frame
    frame_counts = window_df.groupby('time_seconds')['player_id'].count()
    
    # Plot
    plt.figure(figsize=(12, 4))
    plt.plot(frame_counts.index, frame_counts.values, label='Detected players')
    plt.axvline(ready_t, color='orange', linestyle='--', label='Ready time')
    plt.axvline(kick_t, color='red', linestyle='--', label='Kick time')
    plt.xlabel('Time (seconds)')
    plt.ylabel('# Players detected')
    plt.title('Build-Up Window: Player Detection Over Time')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"Setup phase duration: {kick_t - ready_t:.2f}s")
    print(f"Analysis window duration: {window_df['time_seconds'].max() - kick_t:.2f}s")

## Step 6: Batch Extraction

Extract all build-ups from the match using the main extraction pipeline.

In [ ]:
from extraction import extract_match

OUTPUT_DIR = Path("data/processed/rm_pressing_tutorial")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Extract all build-ups
extract_match(
    match_id=MATCH_ID,
    raw_root=RAW_ROOT,
    out_dir=OUTPUT_DIR
)

# Load index
index = pd.read_parquet(OUTPUT_DIR / "index.parquet")
print(f"\nExtracted {len(index)} build-ups total")
print("\nIndex preview:")
print(index.head())

## Step 7: Inspect Extracted Files

Each build-up is saved as a separate parquet file with metadata.

In [ ]:
# List extracted files
frames_dir = OUTPUT_DIR / "frames"
metadata_dir = OUTPUT_DIR / "metadata"

print(f"Frames directory: {len(list(frames_dir.glob('*.parquet')))} files")
print(f"Metadata directory: {len(list(metadata_dir.glob('*.json')))} files")

# Load and inspect first build-up
first_build_up_id = index.iloc[0]['build_up_id']
frames_file = frames_dir / f"build_up_{first_build_up_id:05d}.parquet"
meta_file = metadata_dir / f"build_up_{first_build_up_id:05d}.json"

df = pd.read_parquet(frames_file)
with open(meta_file) as f:
    meta = json.load(f)

print(f"\nBuild-up {first_build_up_id}:")
print(f"  Frames: {len(df)}")
print(f"  Columns: {df.columns.tolist()}")
print(f"\n  Metadata:")
for k, v in meta.items():
    print(f"    {k}: {v}")

## Summary

You've learned how to:
1. ✅ Load SkillCorner match data
2. ✅ Detect opponent goal-kick events
3. ✅ Extract build-up tracking windows
4. ✅ Understand the three-phase structure (setup → kick → post-kick)
5. ✅ Batch-extract all build-ups from a match

## Next Steps

- **Tutorial 2**: Feature Engineering - Extract pressing metrics from build-ups
- **Tutorial 3**: Model Training - Learn spatial zones with GMM and discover topics with NMF
- **Tutorial 4**: Visualization - Create animations and heatmaps

## Troubleshooting

Common issues:
- **Empty index**: No goal kicks detected → Check that data is for opponent goal kicks (not Real Madrid's)
- **Missing tracking**: Frames not found → Verify tracking data quality and frame alignment
- **Team ID error**: Can't identify Real Madrid → Update team name patterns in `team_utils.py`